In [9]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

In [10]:
# Setting seed for reproducibility
np.random.seed(42)

In [11]:
# --- Dim_Center ---
centers = [
    {'Center_ID': 'C001', 'Center_Name': 'CPD Delhi Apex Lab', 'Region': 'North'},
    {'Center_ID': 'C002', 'Center_Name': 'CPD Mumbai Andheri Zone', 'Region': 'West'},
    {'Center_ID': 'C003', 'Center_Name': 'CPD Kolkata Logistics Hub', 'Region': 'East'}, 
    {'Center_ID': 'C004', 'Center_Name': 'CPD Bengaluru Tech Unit', 'Region': 'South'},
    {'Center_ID': 'C005', 'Center_Name': 'CPD Chandigarh Satellite', 'Region': 'North'}
]

df_dim_center = pd.DataFrame(centers)

In [12]:
# --- Dim_TestType ---
test_types = [
    {'TestType_ID': 'T01', 'TestType_Name': 'CBC (Blood)', 'Category': 'Hematology', 'Expected_TAT_Minutes': 60},
    {'TestType_ID': 'T02', 'TestType_Name': 'X-Ray Chest', 'Category': 'Radiology', 'Expected_TAT_Minutes': 120},
    {'TestType_ID': 'T03', 'TestType_Name': 'MRI Brain', 'Category': 'Radiology', 'Expected_TAT_Minutes': 240},
    {'TestType_ID': 'T04', 'TestType_Name': 'Lipid Profile', 'Category': 'Biochemistry', 'Expected_TAT_Minutes': 90},
    {'TestType_ID': 'T05', 'TestType_Name': 'COVID RT-PCR', 'Category': 'Virology', 'Expected_TAT_Minutes': 360}
]
df_dim_testtype = pd.DataFrame(test_types)

In [13]:
# --- Dim_Status ---
statuses = [
    {'Status_ID': 'S01', 'Status_Name': 'Registered', 'Is_Final_Status': False},
    {'Status_ID': 'S02', 'Status_Name': 'Sample Collected', 'Is_Final_Status': False},
    {'Status_ID': 'S03', 'Status_Name': 'Processing', 'Is_Final_Status': False},
    {'Status_ID': 'S04', 'Status_Name': 'Report Generating', 'Is_Final_Status': False},
    {'Status_ID': 'S05', 'Status_Name': 'Completed', 'Is_Final_Status': True},
    {'Status_ID': 'S06', 'Status_Name': 'Cancelled', 'Is_Final_Status': True}
]
df_dim_status = pd.DataFrame(statuses)

In [14]:
# --- Dim_Patient ---
# UPDATED: Increased to 3,500 Patients
patient_ids = [f'P{i:05d}' for i in range(1, 3501)]
ages = np.random.randint(1, 90, 3500)
genders = np.random.choice(['Male', 'Female'], 3500)

def get_age_band(age):
    if age < 18: return '0-18'
    elif age < 35: return '19-35'
    elif age < 60: return '36-60'
    else: return '60+'

df_dim_patient = pd.DataFrame({
    'Patient_ID': patient_ids,
    'Age': ages,
    'Gender': genders,
    'Age_Band': [get_age_band(a) for a in ages]
})

In [15]:
# ==========================================
# 2. GENERATE FACT TABLE (1 Year) - REVISED FOR 15% FAIL RATE
# ==========================================

start_date = datetime(2025, 1, 1)
end_date = datetime(2025, 12, 31)
date_range = pd.date_range(start_date, end_date).tolist()

records = []
counter = 1

for day in date_range:
    # Seasonality
    if day.month in [11, 12, 1]:
        daily_tests = np.random.randint(55, 80)
    else:
        daily_tests = np.random.randint(40, 60)
    
    for _ in range(daily_tests):
        test_id = f'TEST-{counter:07d}'
        patient = random.choice(patient_ids)
        test_meta = random.choice(test_types)
        
        # Traffic Weighting 
        proc_center = np.random.choice(centers, p=[0.20, 0.20, 0.25, 0.20, 0.15])
        
        # Mismatch Logic (12% chance)
        if random.random() > 0.12:
            act_center = proc_center
            match_flag = 'Match'
        else:
            act_center = random.choice([c for c in centers if c != proc_center])
            match_flag = 'Mismatch'

        # Time Logic 
        start_hour = np.random.randint(7, 19)
        stage1_start = day.replace(hour=start_hour, minute=np.random.randint(0, 59))
        
        # --- DELAYS (Keep your requested heavy delays) ---
        # These are large delays, but now they are applied to a faster base process
        delay_factor = 25 if match_flag == 'Mismatch' else 0
        slowness_factor = 15 if proc_center['Center_ID'] == 'C003' else 0

        # --- UPDATED DURATIONS (The Fix) ---
        # S1: Sample Collection (Mean 12 mins)
        dur_s1 = max(5, int(np.random.normal(12, 4))) + delay_factor 
        
        # S2: Processing (Mean 25 mins - WAS 45)
        # We reduced this so "Healthy" tests pass easily. 
        # Only "Delayed" tests will fail.
        dur_s2 = max(10, int(np.random.normal(25, 8))) + slowness_factor
        
        # S3: Reporting (Mean 10 mins - WAS 20)
        dur_s3 = max(5, int(np.random.normal(10, 3)))

        # Gaps between stages (Transfer time)
        gap_1 = np.random.randint(2, 10)
        gap_2 = np.random.randint(2, 10)

        stage1_end = stage1_start + timedelta(minutes=dur_s1)
        stage2_start = stage1_end + timedelta(minutes=gap_1) 
        stage2_end = stage2_start + timedelta(minutes=dur_s2)
        stage3_start = stage2_end + timedelta(minutes=gap_2)
        stage3_end = stage3_start + timedelta(minutes=dur_s3)

        # Status Logic
        rand_status = random.random()
        if rand_status > 0.05:
            status_id = 'S05' # Completed
            tat_missing_flag = False
            total_tat = (stage3_end - stage1_start).total_seconds() / 60
        elif rand_status > 0.02:
            status_id = 'S06' # Cancelled
            tat_missing_flag = True
            total_tat = np.nan
        else:
            status_id = 'S03' # Stuck
            tat_missing_flag = True
            total_tat = np.nan
            stage3_start, stage3_end = np.nan, np.nan 

        # Calculate Flags
        if not tat_missing_flag:
            exp_tat = test_meta['Expected_TAT_Minutes']
            under_tat = total_tat <= exp_tat
            out_tat = total_tat > exp_tat
        else:
            under_tat = False
            out_tat = False

        records.append({
            'Test_ID': test_id,
            'Test_Date': day.date(),
            'Patient_ID': patient,
            'Processing_Center_ID': proc_center['Center_ID'],
            'Actual_Test_Center_ID': act_center['Center_ID'],
            'TestType_ID': test_meta['TestType_ID'],
            'Status_ID': status_id,
            'Stage1_Start': stage1_start,
            'Stage1_End': stage1_end,
            'Stage2_Start': stage2_start,
            'Stage2_End': stage2_end,
            'Stage3_Start': stage3_start,
            'Stage3_End': stage3_end,
            'Total_TAT_Minutes': total_tat,
            'Expected_TAT_Minutes': test_meta['Expected_TAT_Minutes'],
            'Under_TAT_Flag': under_tat,
            'Out_TAT_Flag': out_tat,
            'TAT_Missing_Flag': tat_missing_flag,
            'Center_Match_Flag': match_flag
        })
        counter += 1

df_fact = pd.DataFrame(records)
print(f"New Data Generated. Rows: {len(df_fact)}")

New Data Generated. Rows: 19915


In [16]:
# ==========================================
# 3. EXPORT 
# ==========================================
df_fact.to_csv('Fact_Test_Performance.csv', index=False)
df_dim_center.to_csv('Dim_Center.csv', index=False)
df_dim_patient.to_csv('Dim_Patient.csv', index=False)
df_dim_testtype.to_csv('Dim_TestType.csv', index=False)
df_dim_status.to_csv('Dim_Status.csv', index=False)

print(f"Dataset Generated Successfully.")
print(f"Total Tests: {len(df_fact)}")

Dataset Generated Successfully.
Total Tests: 19915
